# 04. 모델 평가 (Evaluation)
**Credit Risk Dataset — 대출 금리 예측 프로젝트**

---
### 평가 항목
1. 저장된 모델 로드 및 테스트 세트 재구성
2. 회귀 모델 최종 평가 (RMSE·MAE·R²)
3. 잔차(Residual) 분석
4. 피처 중요도 분석
5. 예측값 vs 실제값 산점도
6. 분류 모델 평가 (Precision·Recall·AUC-ROC)
7. 종합 결론

---
## 0. 환경 설정

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
)

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

ROOT           = os.path.abspath('..')
PROCESSED_PATH = os.path.join(ROOT, 'data', 'processed', 'credit_risk_cleaned.csv')
RATE_PATH      = os.path.join(ROOT, 'models', 'loan_rate_model.pkl')
STATUS_PATH    = os.path.join(ROOT, 'models', 'loan_status_model.pkl')

SEED         = 42
FEATURES     = [
    'person_age', 'person_income', 'person_home_ownership', 'person_emp_length',
    'loan_intent', 'loan_grade', 'loan_amnt', 'loan_status',
    'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length',
]
FEATURES_CLS = [f for f in FEATURES if f != 'loan_status']

print('설정 완료')

---
## 1. 모델 로드 및 테스트 세트 재구성

In [ ]:
regressor  = joblib.load(RATE_PATH)
classifier = joblib.load(STATUS_PATH)
print(f'회귀 모델: {type(regressor).__name__}')
print(f'분류 모델: {type(classifier).__name__}')

In [ ]:
df = pd.read_csv(PROCESSED_PATH)

X  = df[FEATURES]
y  = df['loan_int_rate']
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

Xc = df[FEATURES_CLS]
yc = df['loan_status']
_, Xc_test, _, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=SEED)

print(f'테스트 세트: {len(X_test):,}행')

---
## 2. 회귀 모델 최종 평가

In [ ]:
y_pred = regressor.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print('='*45)
print(f'회귀 모델 최종 성능 ({type(regressor).__name__})')
print('='*45)
print(f'  RMSE: {rmse:.4f}')
print(f'  MAE:  {mae:.4f}')
print(f'  R²:   {r2:.4f}')
print()
print(f'  해석: 예측 금리가 실제 금리와 평균 {mae:.2f}%p 오차')
print(f'        분산의 {r2*100:.1f}%를 모델이 설명')

---
## 3. 잔차(Residual) 분석

In [ ]:
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(residuals, bins=50, color='#4C72B0', edgecolor='white', linewidth=0.3)
axes[0].axvline(0, color='red', linestyle='--', lw=1.5)
axes[0].set_title('잔차 분포', fontsize=12, fontweight='bold')
axes[0].set_xlabel('잔차 (실제 - 예측)')
axes[0].set_ylabel('빈도')

axes[1].scatter(y_pred, residuals, alpha=0.2, s=6, color='#4C72B0')
axes[1].axhline(0, color='red', linestyle='--', lw=1.5)
axes[1].set_title('잔차 vs 예측값', fontsize=12, fontweight='bold')
axes[1].set_xlabel('예측 금리 (%)')
axes[1].set_ylabel('잔차')

axes[2].scatter(y_test, np.abs(residuals), alpha=0.2, s=6, color='#C44E52')
axes[2].set_title('절대 잔차 vs 실제값', fontsize=12, fontweight='bold')
axes[2].set_xlabel('실제 금리 (%)')
axes[2].set_ylabel('|잔차|')

plt.suptitle('잔차 분석', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'잔차 평균: {residuals.mean():.4f}  표준편차: {residuals.std():.4f}')
print(f'잔차 범위: {residuals.min():.2f} ~ {residuals.max():.2f}')

---
## 4. 피처 중요도 분석

In [ ]:
labels_kr = {
    'person_age':               '나이',
    'person_income':            '연소득',
    'person_home_ownership':    '주거형태',
    'person_emp_length':        '재직기간',
    'loan_intent':              '대출목적',
    'loan_grade':               '대출등급',
    'loan_amnt':                '대출금액',
    'loan_status':              '대출상태',
    'loan_percent_income':      '소득대비비율',
    'cb_person_default_on_file': '부도이력',
    'cb_person_cred_hist_length': '신용이력',
}

feat_imp = pd.Series(
    regressor.feature_importances_, index=FEATURES
).sort_values()
feat_imp.index = [labels_kr.get(i, i) for i in feat_imp.index]

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = ['#C44E52' if v == feat_imp.max() else '#4C72B0' for v in feat_imp.values]
ax.barh(feat_imp.index, feat_imp.values, color=bar_colors, edgecolor='white')
ax.set_xlabel('중요도')
ax.set_title(f'{type(regressor).__name__} — 피처 중요도', fontsize=12, fontweight='bold')
for i, (_, val) in enumerate(feat_imp.items()):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 5. 예측값 vs 실제값 산점도

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred, alpha=0.2, s=10, color='#4C72B0')
lims = [min(y_test.min(), y_pred.min()) - 0.5,
        max(y_test.max(), y_pred.max()) + 0.5]
ax.plot(lims, lims, 'r--', lw=1.5, label='완벽한 예측선 (y=x)')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('실제 대출 금리 (%)')
ax.set_ylabel('예측 대출 금리 (%)')
ax.set_title(f'실제값 vs 예측값 (R² = {r2:.4f})', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 6. 분류 모델 평가 (부도 예측)

In [ ]:
yc_pred      = classifier.predict(Xc_test)
yc_pred_prob = classifier.predict_proba(Xc_test)[:, 1]

print('분류 성능 리포트:')
print(classification_report(yc_test, yc_pred, target_names=['정상(0)','부도(1)']))
print(f'AUC-ROC: {roc_auc_score(yc_test, yc_pred_prob):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(yc_test, yc_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['예측 정상','예측 부도'],
            yticklabels=['실제 정상','실제 부도'])
axes[0].set_title('혼동 행렬', fontsize=12, fontweight='bold')

fpr, tpr, _ = roc_curve(yc_test, yc_pred_prob)
auc_score   = roc_auc_score(yc_test, yc_pred_prob)
axes[1].plot(fpr, tpr, color='#4C72B0', lw=2, label=f'ROC (AUC = {auc_score:.4f})')
axes[1].plot([0,1],[0,1], 'r--', lw=1.2, label='Random (AUC = 0.5)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC 곡선', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('분류 모델 평가 (RandomForestClassifier)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 7. 종합 결론

In [ ]:
auc = roc_auc_score(yc_test, yc_pred_prob)

print('='*55)
print('최종 평가 결론')
print('='*55)
print(f"""
[회귀 — 대출 금리 예측]
  모델:  {type(regressor).__name__} (n_estimators=200)
  RMSE:  {rmse:.4f}  →  예측 오차 평균 약 {rmse:.2f}%p
  MAE:   {mae:.4f}  →  절대 오차 평균 약 {mae:.2f}%p
  R²:    {r2:.4f}  →  금리 분산의 {r2*100:.1f}%를 설명
  판정:  R² 0.90 이상 — 실용적 예측 수준
  주요 피처: 대출 등급 > 소득 대비 비율 > 대출 금액 순

[분류 — 부도 예측]
  모델:  {type(classifier).__name__} (n_estimators=200)
  AUC-ROC: {auc:.4f}  →  우수한 분류 성능

[결론]
  트리 기반 앙상블 모델이 선형 모델 대비 월등히 높은 성능.
  대출 등급(loan_grade)이 금리 결정에 가장 결정적 요소임을 확인.
  실제 금융 서비스 적용 시 등급 산출 로직의 정확도가 핵심 변수.
""")